In [1]:
%cd ../../..

/Users/hoangle/Projects/Food-Waste-Optimization


In [2]:
import time
import random
from itertools import combinations, product
from pathlib import Path

import pandas as pd
import polars as pl

from src.services.pcs_forecast import forecast_pcs_per_meal

2024-12-03 17:43:08.278 | INFO     | src.services.pcs_forecast:<module>:25 - 


In [3]:
# path = "data/processed/phase_4/menus/vik_2024-12-02.parquet"

# df = pd.read_parquet(path)

# df.head()

In [4]:
seed = time.time()
random.seed(seed)

In [5]:
NUM_VEGAN_PER_DAY = 2
NUM_MEALS_PER_DAY = [3, 4]
NUM_DAY_LEVEL_MENUS = 10_000_000
MAX_MEAL_OCCURENCES = 2
MIN_KELA_PER_DAY = 2
NUM_DAY_LEVEL_MENUS_2 = 1000
NUM_EACH_COMBO = 1_000_000
NUM_FISH_PER_WEEK = 2

# restaurant = "phy"
restaurant = "che"
# restaurant = "exa"
# restaurant = "vik"
schoolyear = "24-25"
date_start = "2024-12-02"

In [6]:
# meals = pd.DataFrame({
#     'meal_id': [7614, 9500047, 1307],
#     'restaurant': restaurant,
#     'date': pd.to_datetime(date_start),
# })

# forecast_pcs_per_meal(meals)

# Polars

In [7]:
path = "data/processed/phase_4/dim_meals.xlsx"
dim_meals_raw = pl.read_excel(path)
dim_meals_raw.head()

meal_id,meal_type_1,schoolyear,is_kela,is_new,restaurant,meal_type_2,pcs_mean
i64,str,str,bool,bool,str,str,f64
9017,"""vegan""","""24-25""",true,true,"""che-exa-vik""","""vegan-miscellaneous""",116.3425
7201,"""vegan""","""23-24""",false,false,null,null,106.231119
9032,"""vegan""","""23-24""",false,false,null,null,106.231119
9102,"""vegan""","""23-24""",false,false,null,null,106.231119
7010,"""vegetarian""","""24-25""",false,false,"""che-exa-vik""",null,71.0


In [8]:
path = "data/processed/phase_4/dim_co2.xlsx"
dim_co2 = pl.read_excel(path)
dim_co2.head()

meal_id,co2
i64,f64
34,0.81
37,0.61
710,0.67
713,0.56
724,0.82


In [9]:
path = "data/processed/phase_4/dim_waste.xlsx"
dim_waste = pl.read_excel(path)
dim_waste.head()

meal_id,waste
i64,f64
9017,0.01
7201,0.01
9032,0.01
9102,0.01
7010,0.039256


In [10]:
path = 'data/processed/phase_4/dim_pieces_whole.xlsx'
dim_pcs_whole = pl.read_excel(path)

dim_pcs_whole.head()

date,pcs,restaurant
date,f64,str
2024-11-01,151.66,"""phy"""
2024-11-04,239.91,"""phy"""
2024-11-05,257.39,"""phy"""
2024-11-06,261.16,"""phy"""
2024-11-07,270.72,"""phy"""


# Craft the day-level menus

Day-level menus must satisfy:
- condition (2) and (6)
- containing a variety of meals

In [11]:
# During crafting the menus, the condition (6) are already met

meals = dim_meals_raw.filter(
    pl.col('restaurant').is_not_null(),
    pl.col('restaurant').str.contains(restaurant),
    pl.col('schoolyear') == pl.lit(schoolyear)
)


meals_vegan = meals.filter(pl.col("meal_type_1") == pl.lit("vegan")).select("meal_id")
meals_notvegan = meals.filter(pl.col("meal_type_1") != pl.lit("vegan")).select("meal_id")

list_meals_vegan = meals_vegan.select('meal_id').to_series().to_list()
list_meals_notvegan = meals_notvegan.select('meal_id').to_series().to_list()

vegan_combo2 = list(combinations(list_meals_vegan, 2))
vegan_combo3 = list(combinations(list_meals_vegan, 3))
vegan_combo4 = list(combinations(list_meals_vegan, 4))

notvegan_combo1 = [(x,) for x in list_meals_notvegan]
notvegan_combo2 = list(combinations(list_meals_notvegan, 2))

In [12]:
def _to_list(list1: list, list2: list):
    return [(*x, *y) for (x, y) in product(list1, list2)]
list_combos = []

# Create combo with 2 vegan and 1 nonvegan
list_combos.extend(_to_list(vegan_combo2, notvegan_combo1))

# Create combo with 3 vegan
list_combos.extend(vegan_combo3)

if restaurant in ["che", "exa", 'vik']:
    # Create combo with 2 vegan and 2 nonvegan
    list_combos.extend(_to_list(vegan_combo2, notvegan_combo2))

    # Create combo with 3 vegan and 1 nonvegan
    list_combos.extend(_to_list(vegan_combo3, notvegan_combo1))

    # Create combo with 4 vegan
    list_combos.extend(vegan_combo4)


In [13]:
is_kela = (
    meals
    .select("meal_id", "is_kela")
    # .lazy()
)
meal_type = (
    meals
    .select("meal_id", pl.col("meal_type_1").alias('meal_type'))
    # .lazy()
)

PREFIXES = ["mon", "tue", "wed", "thu", "fri"]

list_menus_week = []
list_menus_days = []
for prefix in PREFIXES:
    menu_daylevel = (
        pl
        .DataFrame({'meal_id': random.sample(list_combos, min(len(list_combos), NUM_EACH_COMBO))})
        # .lazy()
        .with_row_index()
        .explode('meal_id')
    )

    # Filter out day-level menus not satisfying condition (1.2)
    ids_2kela = (
        menu_daylevel.
        join(is_kela, on='meal_id', how='left')
        # .select(
        #     'index', 'meal_id',
            
        # )
        .group_by('index')
        .agg(pl.col('is_kela').cast(pl.Int64).sum())
        .filter(pl.col('is_kela') >= pl.lit(MIN_KELA_PER_DAY))
        .select('index')
    )
    menu_daylevel = menu_daylevel.join(ids_2kela, on='index', how='inner')


    menu_daylevel = menu_daylevel.select(
        pl.col('index').alias('idx_daylevel'),
        pl.col('meal_id'),
        pl.lit(prefix).alias('weekday')
    )

    list_menus_days.append(menu_daylevel)

    list_menus_week.append(
        menu_daylevel
        .unique('idx_daylevel', keep='first', maintain_order=True)
        .with_row_index()
        .select(
            pl.col('index').alias('idx_weeklevel'),
            'idx_daylevel',
            'weekday'
        )
    )

menus_days = pl.concat(list_menus_days)
menus_week = pl.concat(list_menus_week)

In [14]:
# Filter out week-level menus not having all weekdays
ids_fullweek = (
    menus_week
    .group_by('idx_weeklevel')
    .len()
    .filter(pl.col('len') == 5)
    .select('idx_weeklevel')
)
menus_week = menus_week.join(ids_fullweek, on='idx_weeklevel', how='inner')


menus_week = menus_week.join(menus_days, on=['idx_daylevel', 'weekday'], how='left')


# Filter week-level menus not satisfying condition (3)
ids_max_occur = (
    menus_week
    .group_by(['idx_weeklevel', 'meal_id'])
    .len()
    .filter(pl.col('len') <= MAX_MEAL_OCCURENCES)
    .unique('idx_weeklevel')
    .select('idx_weeklevel')
)
menus_week = menus_week.join(ids_max_occur, on='idx_weeklevel', how='inner')

# Filter out week-level menus not satisfying condition (1.1)
ids_fish = (
    menus_week
    .join(meal_type, on='meal_id', how='left')
    .group_by(['idx_weeklevel', 'meal_type'])
    .len()
    .filter(
        (pl.col('meal_type') == pl.lit('fish'))
        & (pl.col('len') == NUM_FISH_PER_WEEK)
    )
    .unique('idx_weeklevel')
    .select('idx_weeklevel')
)
menus_week = menus_week.join(ids_fish, on='idx_weeklevel', how='inner')


# Limit the max number of week-level menus  
ids_max_week = menus_week.unique('idx_weeklevel').select('idx_weeklevel').head(NUM_DAY_LEVEL_MENUS_2)
menus_week = menus_week.join(ids_max_week, on='idx_weeklevel', how='inner')


menus_week = (
    menus_week
    .select(
        pl.col('idx_weeklevel').alias('index'),
        'weekday',
        'meal_id'
    )
)


menus_week.head()

index,weekday,meal_id
u32,str,i64
289,"""mon""",200005
289,"""mon""",1307
289,"""mon""",9018
289,"""mon""",6546
402,"""mon""",6088


In [15]:
# (
#     menus_week
#     # .filter(pl.col('index') == 33)
#     # .group_by('weekday')
#     .join(meals.select('meal_id', 'meal_type_1'), on='meal_id', how='left')
#     .filter(pl.col('meal_type_1') == pl.lit('fish'))
#     .group_by('index', 'meal_type_1')
#     .len()
#     .filter(pl.col('len') != 2)
# )

## Add predicted info (biowate, pcs, CO2, pcs_whole)

In [16]:
weekday2date = pl.DataFrame({
    'weekday': PREFIXES,
    'date': pd.date_range(date_start, periods=5)
})

In [17]:
meals_pred_info = (
    menus_week
    .join(weekday2date, on='weekday', how='left')
    .filter(pl.col('meal_id') != pl.lit(-1))
    .select(
        'index',
        'meal_id',
        pl.col('date').dt.date(),
        pl.lit(restaurant).alias('restaurant')
    )
)

meals_pred_info.head()

index,meal_id,date,restaurant
u32,i64,date,str
289,200005,2024-12-02,"""che"""
289,1307,2024-12-02,"""che"""
289,9018,2024-12-02,"""che"""
289,6546,2024-12-02,"""che"""
402,6088,2024-12-02,"""che"""


In [18]:
# Add predicted POS
pcs_pred = pl.from_pandas(forecast_pcs_per_meal(meals_pred_info.to_pandas()))
pcs_pred = pcs_pred.select(
    pl.col('index').cast(pl.UInt32),
    pl.col('date').dt.date(),
    'restaurant', 'meal_id', 'pcs_pred'
    
)

meals_pred_info = meals_pred_info.join(pcs_pred, on=['index', 'date', 'restaurant', 'meal_id'], how='left')



# Add CO2
meals_pred_info = meals_pred_info.join(dim_co2, on='meal_id', how='left')


# Add biowaste
meals_pred_info = meals_pred_info.join(dim_waste, on='meal_id', how='left')



meals_pred_info.head()

index,meal_id,date,restaurant,pcs_pred,co2,waste
u32,i64,date,str,f32,f64,f64
289,200005,2024-12-02,"""che""",74.894287,0.49,7.36678
289,1307,2024-12-02,"""che""",127.825783,0.45,0.037464
289,9018,2024-12-02,"""che""",75.250038,0.42,0.169506
289,6546,2024-12-02,"""che""",130.387192,0.34,0.03196
402,6088,2024-12-02,"""che""",158.469208,0.46,0.104143


# Calculate fitness value

In [19]:
THETA_CO2 = 0.5
THETA_WASTE = 0.04
ALPHA_PCS = 2
ALPHA_CO2 = 1
ALPHA_WASTE = 1

In [20]:
menus_fitness = (
    meals_pred_info
    .with_columns(
        (pl.col('pcs_pred') * pl.col('co2')).alias('co2_total_pred'),
        (pl.col('pcs_pred') * pl.col('waste')).alias('waste_total_pred'),
    )
    .group_by(['restaurant', 'date', 'index']).agg(
        pl.col('meal_id').alias('meal_ids'),
        pl.col('pcs_pred').sum().alias('pcs_sum_pred'),
        pl.col('co2_total_pred').sum().alias('co2_sum_pred'),
        pl.col('waste_total_pred').sum().alias('waste_sum_pred')
    )
    .join(dim_pcs_whole, on=['date', 'restaurant'], how='left').rename({'pcs': 'pcs_whole_pred'})
    .with_columns(
        (
            ALPHA_PCS * (pl.col('pcs_sum_pred') / (pl.col('pcs_whole_pred') + 1) - 1).abs()
            + ALPHA_CO2 * (pl.col('co2_sum_pred') / pl.col('pcs_sum_pred')) / THETA_CO2
            + ALPHA_WASTE * (pl.col('waste_sum_pred') / pl.col('pcs_sum_pred')) / THETA_WASTE
        ).alias('fitness')
    )
)

menus_fitness.sort('fitness').head()

restaurant,date,index,meal_ids,pcs_sum_pred,co2_sum_pred,waste_sum_pred,pcs_whole_pred,fitness
str,date,u32,list[i64],f32,f64,f64,f64,f64
"""che""",2024-12-05,404009,"[9050, 9500020, … 8991]",341.551666,60.460151,3.415517,886.54,1.834373
"""che""",2024-12-05,357224,"[6853, 9500150, … 8991]",508.343079,138.48225,9.942953,886.54,1.888316
"""che""",2024-12-03,100936,"[9061, 9060, … 8993]",440.453705,157.462199,4.404537,923.3,2.011946
"""che""",2024-12-03,281973,"[9111, 7583, … 9086]",537.128174,262.254042,5.662167,923.3,2.077806
"""che""",2024-12-02,167166,"[6341, 7595, … 1107]",294.526245,65.818823,2.945262,977.8,2.095136


# To file

In [27]:
(
    menus_fitness
    .filter(pl.col('index') == 404009)
    .explode('meal_ids')
    .join(meals.select('meal_id', 'meal_type_1'), right_on='meal_id', left_on='meal_ids', how='left')
    .filter(pl.col('meal_type_1') == pl.lit('fish'))
    .group_by('index', 'meal_type_1')
    .len()
    # .filter(pl.col('len') != 2)
)

index,meal_type_1,len
u32,str,u32
404009,"""fish""",2


In [ ]:
# path = Path("data/processed/phase_4/menus") / f"{restaurant}_{date_start}.parquet"
# menus_fitness.select('date', 'restaurant', 'meal_ids', 'fitness').write_parquet(path)

# Spark

In [ ]:
# spark = (
#     SparkSession.builder.master("local[*]")
#     .appName("YLVA")
#     .config("spark.sql.execution.arrow.pyspark.enabled", "true")
#     # .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4")
#     # .config("spark.hadoop.fs.s3a.access.key", os.environ.get("AWS_ACCESS_KEY_ID"))
#     # .config(
#     #     "spark.hadoop.fs.s3a.secret.key", os.environ.get("AWS_SECRET_ACCESS_KEY")
#     # )
#     .config("spark.driver.memory", f"{4}g")
#     .config("spark.executor.memory", f"{8}g")
#     .config("spark.dynamicAllocation.enabled", "true")
#     .config("spark.shuffle.service.enabled", "true")
#     .getOrCreate()
# )


## Read data

In [ ]:
# path = "data/processed/phase_4/dim_meals.xlsx"
# dim_meals_raw = spark.createDataFrame(pd.read_excel(path))

# dim_meals_raw.show()

In [ ]:
# def forecast_pos(meal_ids: list[int], restaurant: str, date: str) -> dict:
#     feat = dim_meals[dim_meals['meal_id'].isin(meal_ids)][['meal_id', 'meal_type_1']].copy()
#     feat.columns = ['meal_id', 'meal_type']

#     feat['meal_type'] = feat['meal_type'].map({
#         'meat':    1, # 'meat',
#         'fish':    2, # 'fish',
#         'vegan':    3, # 'vegan',
#         'vegetarian':    4, # 'vegetarian',
#         'chicken':    5, # 'chicken'
#     })


#     feat['restaurant'] = restaurant

#     THETA = 5
#     records = []
#     for r in feat.itertuples():
#         ids = set(meal_ids)
#         ids.remove(r.meal_id)
        
#         for tup in permutations(ids):
#             tup = [*tup]

#             # pad
#             if len(tup) < THETA - 1:
#                 tup.extend([0]*(THETA - 1 - len(tup)))
            
#             records.append({
#                 'date': date,
#                 'restaurant': r.restaurant,
#                 'meal_id': r.meal_id,
#                 'meal_id_other1': tup[0],
#                 'meal_id_other2': tup[1],
#                 'meal_id_other3': tup[2],
#                 'meal_id_other4': tup[3],
#             })

#     feat = pd.DataFrame.from_records(records)



#     def _encode_date_cyclic(t, period_week: int = 7, period_day: int = 31, period_month: int = 12):
#         def get_sin_encoding(x, period: int):
#             return np.sin(2 * np.pi * x / period)
#         def get_cos_encoding(x, period: int):
#             return np.cos(2 * np.pi * x / period)  

#         return pd.Series({
#             'weekday_sin': get_sin_encoding(t.weekday(), period_week),
#             'weekday_cos': get_cos_encoding(t.weekday(), period_week),
#             'day_sin': get_sin_encoding(t.day, period_day),
#             'day_cos': get_cos_encoding(t.day, period_day),
#             'month_sin': get_sin_encoding(t.month, period_month),
#             'month_cos': get_cos_encoding(t.month, period_month),
#         })

#     datetime_encoded = feat['date'].apply(_encode_date_cyclic)
#     feat = pd.concat([feat, datetime_encoded], axis=1)


#     feat = (
#         feat
#         .merge(meal_info, how='left', left_on=['meal_id'], right_on=['id'])
#         .drop(columns='id')
#         .rename(columns={'type': 'meal_type', 'mean': 'pcs_mean'})

#         .merge(meal_info, how='left', left_on=['meal_id_other1'], right_on=['id'])
#         .drop(columns='id')
#         .rename(columns={'type': 'meal_type_other1', 'mean': 'pcs_mean_other1'})

#         .merge(meal_info, how='left', left_on=['meal_id_other2'], right_on=['id'])
#         .drop(columns='id')
#         .rename(columns={'type': 'meal_type_other2', 'mean': 'pcs_mean_other2'})

#         .merge(meal_info, how='left', left_on=['meal_id_other3'], right_on=['id'])
#         .drop(columns='id')
#         .rename(columns={'type': 'meal_type_other3', 'mean': 'pcs_mean_other3'})

#         .merge(meal_info, how='left', left_on=['meal_id_other4'], right_on=['id'])
#         .drop(columns='id')
#         .rename(columns={'type': 'meal_type_other4', 'mean': 'pcs_mean_other4'})
#     )
#     feat = feat[~feat['pcs_mean'].isna()].fillna(0)         # Filter out records having no `pcs_mean` and impute data


#     # Encode meal_id

#     feat['meal_id_enc'] = enc_meal_id.transform(feat[['meal_id']])

#     cols = ['meal_id_other1', 'meal_id_other2', 'meal_id_other3', 'meal_id_other4']
#     for col in cols:
#         encoded = enc_meal_id.transform(feat[[col]])
#         mask = (feat[col] != 0).astype(np.int32)
#         encoded = encoded.squeeze() * mask

#         feat[f'{col}_enc'] = encoded


#     # Assign categorical column type
#     cols_cat = [
#         'restaurant',
#         'meal_type',
#         'meal_type_other1',
#         'meal_type_other2',
#         'meal_type_other3',
#         'meal_type_other4',
#     ]
#     for col in cols_cat:
#         feat[col] = feat[col].astype('category')

#     # Keep important columns
#     cols_X = [
#         'weekday_sin',
#         'weekday_cos',
#         'day_sin',
#         'day_cos',
#         'month_sin',
#         'month_cos',

#         'restaurant',

#         'meal_id_enc',
#         'meal_type',

#         'pcs_mean',


#         'meal_id_other1_enc',
#         'meal_id_other2_enc',
#         'meal_id_other3_enc',
#         'meal_id_other4_enc',

#         'meal_type_other1',
#         'meal_type_other2',
#         'meal_type_other3',
#         'meal_type_other4',

#         'pcs_mean_other1',
#         'pcs_mean_other2',
#         'pcs_mean_other3',
#         'pcs_mean_other4',
#     ]
#     X = feat[cols_X]

#     X.head()

## Generate menu candidates

In [ ]:
# meals = (
#     dim_meals_raw
#     .filter(
#         (dim_meals_raw.restaurant.isNotNull())
#         & (dim_meals_raw.restaurant.contains(restaurant))
#         & (dim_meals_raw.schoolyear == schoolyear)
#     )
#     .withColumn("id", pF.monotonically_increasing_id())
# )

### 1. Generate day-level menus

In [ ]:
# # Compose vegan combo
# meals_vegan = meals.filter(meals.meal_type_1 == "vegan").select("id", "meal_id")
# combo_vegan = (
#     meals_vegan
#     .select(
#         pF.col('meal_id').alias('meal_id_1'),
#         pF.col('id').alias('id_1')
#     )
#     .crossJoin(meals_vegan)
#     .filter(pF.col("id_1") < pF.col("id"))
#     .select(
#         pF.col("meal_id_1").alias("vegan_1"),
#         pF.col("meal_id").alias("vegan_2"),
#     )
# )



# # Compose non-vegan combo
# meals_notvegan = meals.filter(meals.meal_type_1 != "vegan").select("id", "meal_id")

# meals_notvegan_single = meals_notvegan.select(
#     pF.col('meal_id').alias("nonvegan_1"),
#     pF.lit(-1).alias("nonvegan_2"),
# )
# meals_notvegan_multi = (
#     meals_notvegan
#     .select(
#         pF.col('meal_id').alias('meal_id_1'),
#         pF.col('id').alias('id_1')
#     )
#     .crossJoin(meals_notvegan)
#     .filter(pF.col("id_1") < pF.col("id"))
#     .select(
#         pF.col("meal_id_1").alias("nonvegan_1"),
#         pF.col("meal_id").alias("nonvegan_2"),
#     )
# )
# combo_notvegan = reduce(DataFrame.unionAll, [meals_notvegan_single, meals_notvegan_multi])



# # Compose combo
# is_kela = meals.select("meal_id", "is_kela")

# menus_day = (
#     combo_vegan
#     .crossJoin(combo_notvegan)
#     .join(
#         meals.select(
#             pF.col("meal_id").alias("vegan_1"),
#             pF.when(pF.col("is_kela"), 1).otherwise(0).alias("kela_vegan_1")
#         ),
#         on='vegan_1',
#         how='left'
#     )
#     .join(
#         meals.select(
#             pF.col("meal_id").alias("vegan_2"),
#             pF.when(pF.col("is_kela"), 1).otherwise(0).alias("kela_vegan_2")
#         ),
#         on='vegan_2',
#         how='left'
#     )
#     .join(
#         meals.select(
#             pF.col("meal_id").alias("nonvegan_1"),
#             pF.when(pF.col("is_kela"), 1).otherwise(0).alias("kela_nonvegan_1")
#         ),
#         on='nonvegan_1',
#         how='left'
#     )
#     .join(
#         meals.select(
#             pF.col("meal_id").alias("nonvegan_2"),
#             pF.when(pF.col("is_kela"), 1).otherwise(0).alias("kela_nonvegan_2")
#         ),
#         on='nonvegan_2',
#         how='left'
#     )
#     .filter(pF.col('kela_vegan_1') + pF.col('kela_vegan_2') + pF.col('kela_nonvegan_1') + pF.col('kela_nonvegan_2') >= MIN_KELA_PER_DAY)
#     .select(
#         "nonvegan_2", "nonvegan_1", "vegan_2", "vegan_1"
#     )
#     .orderBy(pF.rand())
# )

# # menus_day.show()

In [ ]:
# PREFIXES = ["mon", "tue", "wed", "thu", "fri"]

# menus_week = None

# for prefix in PREFIXES:
#     tmp = (
#         menus_day
#         .sample(0.2, np.random.randint(0, 10000))
#         .limit(NUM_DAY_LEVEL_MENUS)
#         .select([pF.col(c).alias(f"{c}_{prefix}") for c in menus_day.columns])
#     )

#     if menus_week is None:
#         menus_week = tmp
#     else:
#         menus_week = menus_week.crossJoin(tmp)

# cols = menus_week.columns
# menus_week = menus_week.withColumn("id", pF.monotonically_increasing_id())

# menus_id_valid = (
#     menus_week
#     .melt(ids='id', values=cols, variableColumnName='type', valueColumnName="meal_id")
#     .groupBy('id', 'meal_id')
#     .count()
#     .groupBy('id')
#     .agg(pF.max("count").alias("max_per_meal"))
#     .filter(pF.col('max_per_meal') <= MAX_MEAL_OCCURENCES)
#     .select('id')
# )


# menus_week = menus_week.join(menus_id_valid, on='id', how='inner')

# # menus_week.show(truncate=False)

## Write to Parquet

In [ ]:
# path_out = f"data/processed/phase_4/menus/{date}"

# menus_week.write.parquet(path_out)

In [ ]:
# df = spark.read.parquet("data/processed/phase_4/menus/2024-12-02")

# df.show()